In [ ]:
import pandas as pd
import numpy as np
from google.colab import drive
import os
import random
from typing import Dict, List, Tuple

class DatasetCombiner:
    def __init__(self, drive_folder_path: str):
        """
        Initialize the DatasetCombiner with the Google Drive folder path

        Args:
            drive_folder_path: Path to the folder containing CSV files in Google Drive
        """
        self.drive_folder_path = drive_folder_path
        self.csv_files = []
        self.datasets = {}

    def mount_drive(self):
        """Mount Google Drive in Colab"""
        print("Mounting Google Drive...")
        drive.mount('/content/drive')
        print("Google Drive mounted successfully!")

    def discover_csv_files(self):
        """Discover all CSV files in the specified folder"""
        print(f"Discovering CSV files in: {self.drive_folder_path}")

        if not os.path.exists(self.drive_folder_path):
            raise FileNotFoundError(f"Folder not found: {self.drive_folder_path}")

        self.csv_files = [f for f in os.listdir(self.drive_folder_path)
                         if f.lower().endswith('.csv')]

        if not self.csv_files:
            raise FileNotFoundError("No CSV files found in the specified folder")

        print(f"Found {len(self.csv_files)} CSV files:")
        for i, file in enumerate(self.csv_files, 1):
            print(f"  {i}. {file}")

        return self.csv_files

    def load_datasets(self):
        """Load all CSV datasets into memory"""
        print("\nLoading datasets...")

        for file in self.csv_files:
            file_path = os.path.join(self.drive_folder_path, file)
            try:
                df = pd.read_csv(file_path)
                self.datasets[file] = df
                print(f"  ✓ Loaded {file}: {len(df)} rows, {len(df.columns)} columns")
            except Exception as e:
                print(f"  ✗ Error loading {file}: {str(e)}")

        return self.datasets

    def get_dataset_info(self):
        """Display information about loaded datasets"""
        print("\n" + "="*60)
        print("DATASET INFORMATION")
        print("="*60)

        for file, df in self.datasets.items():
            print(f"\nFile: {file}")
            print(f"  Rows: {len(df):,}")
            print(f"  Columns: {len(df.columns)}")
            print(f"  Columns: {list(df.columns)}")
            print(f"  Sample data:")
            print(f"    {df.head(2).to_string(index=False)}")

    def create_combined_dataset(self,
                               percentages: Dict[str, float],
                               total_rows: int = None,
                               random_seed: int = 42) -> pd.DataFrame:
        """
        Create a combined dataset with specified percentages from each file

        Args:
            percentages: Dictionary mapping filename to percentage (0-100)
            total_rows: Total number of rows in final dataset (if None, uses sum of calculated rows)
            random_seed: Seed for reproducible random sampling

        Returns:
            Combined pandas DataFrame
        """
        # Set random seed for reproducibility
        random.seed(random_seed)
        np.random.seed(random_seed)

        # Validate percentages
        total_percentage = sum(percentages.values())
        if abs(total_percentage - 100) > 0.01:
            print(f"Warning: Percentages sum to {total_percentage}%, not 100%")

        # Validate file names
        for file in percentages.keys():
            if file not in self.datasets:
                raise ValueError(f"File '{file}' not found in loaded datasets")

        combined_data = []
        actual_rows = {}

        print(f"\nCreating combined dataset with seed: {random_seed}")
        print("-" * 50)

        for file, percentage in percentages.items():
            if percentage <= 0:
                continue

            df = self.datasets[file]

            if total_rows:
                # Calculate rows based on total_rows and percentage
                rows_to_sample = int((percentage / 100) * total_rows)
            else:
                # Use percentage of available data
                rows_to_sample = int((percentage / 100) * len(df))

            # Ensure we don't sample more rows than available
            rows_to_sample = min(rows_to_sample, len(df))

            if rows_to_sample > 0:
                # Randomly sample rows
                sampled_df = df.sample(n=rows_to_sample, random_state=random_seed)

                # Add source file information
                sampled_df = sampled_df.copy()
                sampled_df['source_file'] = file
                sampled_df['source_percentage'] = percentage

                combined_data.append(sampled_df)
                actual_rows[file] = rows_to_sample

                print(f"  {file}: {rows_to_sample:,} rows ({percentage}%)")

        if not combined_data:
            raise ValueError("No data was sampled. Check your percentages.")

        # Combine all sampled data
        combined_df = pd.concat(combined_data, ignore_index=True)

        # Shuffle the combined dataset
        combined_df = combined_df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

        print(f"\nCombined dataset created:")
        print(f"  Total rows: {len(combined_df):,}")
        print(f"  Total columns: {len(combined_df.columns)}")

        return combined_df

    def save_combined_dataset(self,
                             combined_df: pd.DataFrame,
                             output_filename: str = "combined_dataset.csv"):
        """Save the combined dataset to Google Drive"""
        output_path = os.path.join(self.drive_folder_path, output_filename)

        try:
            combined_df.to_csv(output_path, index=False)
            print(f"\n✓ Combined dataset saved as: {output_filename}")
            print(f"  Location: {output_path}")
            print(f"  Size: {len(combined_df):,} rows × {len(combined_df.columns)} columns")
        except Exception as e:
            print(f"✗ Error saving file: {str(e)}")

# =============================================================================
# USAGE EXAMPLE
# =============================================================================

def main():
    # Configuration - UPDATE THESE VALUES
    DRIVE_FOLDER_PATH = "/content/drive/MyDrive/FYP/"  # Update this path

    # Dataset percentages - UPDATE THESE VALUES
    # The keys should match your CSV filenames
    """DATASET_PERCENTAGES = {
        "web-ids23_benign.csv": 80.0,          # 40% from benign data
        "web-ids23_bruteforce_http.csv": 8.0,  # 15% from bruteforce http
        "web-ids23_bruteforce_https.csv": 8.0, # 15% from bruteforce https
        "web-ids23_dos_http.csv": 3.0,        # 20% from dos http
        "web-ids23_xss_https.csv": 1.0        # 10% from xss https
    }"""

    DATASET_PERCENTAGES = {
        "web-ids23_benign.csv": 75.0,                    # Reduced to 75%

        # Reconnaissance (5% total)
        "web-ids23_portscan.csv": 3.0,                  # 3% - Very common reconnaissance
        "web-ids23_hostsweep_Pn.csv": 1.0,              # 1% - Network discovery
        "web-ids23_hostsweep_sn.csv": 1.0,              # 1% - Network discovery

        # High-frequency attacks (12% total)
        "web-ids23_sql_injection_http.csv": 4.0,        # 4% - Very common
        "web-ids23_sql_injection_https.csv": 3.0,       # 3% - Very common
        "web-ids23_xss_http.csv": 2.5,                  # 2.5% - Very common
        "web-ids23_xss_https.csv": 2.5,                 # 2.5% - Very common

        # Medium-frequency attacks (5% total)
        "web-ids23_bruteforce_http.csv": 1.5,           # 1.5% - Common
        "web-ids23_bruteforce_https.csv": 1.5,          # 1.5% - Common
        "web-ids23_dos_http.csv": 1.0,                  # 1% - Moderate
        "web-ids23_dos_https.csv": 1.0,                 # 1% - Moderate

        # Advanced/Specialized attacks (3% total)
        "web-ids23_revshell_http.csv": 0.5,             # 0.5% - Critical but rare
        "web-ids23_revshell_https.csv": 0.5,            # 0.5% - Critical but rare
        "web-ids23_ssrf_http.csv": 0.5,                 # 0.5% - Growing threat
        "web-ids23_ssrf_https.csv": 0.5,                # 0.5% - Growing threat
        "web-ids23_ssh_login.csv": 0.5,                 # 0.5% - Still relevant
        "web-ids23_ftp_login.csv": 0.3,                 # 0.3% - Legacy
        "web-ids23_smtp_enum.csv": 0.2,                 # 0.2% - Rare
        # ... other specialized attacks with small percentages
    }

    # Optional: Set total number of rows in final dataset
    TOTAL_ROWS = 100000  # Set to None to use percentage of available data

    # Output filename
    OUTPUT_FILENAME = "combined_security_dataset.csv"

    # Random seed for reproducibility
    RANDOM_SEED = 42

    # =============================================================================

    try:
        # Initialize the combiner
        combiner = DatasetCombiner(DRIVE_FOLDER_PATH)

        # Mount Google Drive
        combiner.mount_drive()

        # Discover and load CSV files
        combiner.discover_csv_files()
        combiner.load_datasets()

        # Display dataset information
        combiner.get_dataset_info()

        # Create combined dataset
        combined_df = combiner.create_combined_dataset(
            percentages=DATASET_PERCENTAGES,
            total_rows=TOTAL_ROWS,
            random_seed=RANDOM_SEED
        )

        # Display sample of combined data
        print("\n" + "="*60)
        print("SAMPLE OF COMBINED DATASET")
        print("="*60)
        print(combined_df.head(10))

        print(f"\nSource file distribution:")
        print(combined_df['source_file'].value_counts())

        # Save the combined dataset
        combiner.save_combined_dataset(combined_df, OUTPUT_FILENAME)

        print(f"\n🎉 Process completed successfully!")

    except Exception as e:
        print(f"❌ Error: {str(e)}")

# =============================================================================
# ALTERNATIVE QUICK USAGE FUNCTIONS
# =============================================================================

def quick_combine(folder_path: str, percentages: Dict[str, float], output_name: str = "combined_data.csv"):
    """
    Quick function to combine datasets with minimal setup

    Args:
        folder_path: Path to Google Drive folder with CSV files
        percentages: Dictionary of filename -> percentage
        output_name: Name of output file
    """
    combiner = DatasetCombiner(folder_path)
    combiner.mount_drive()
    combiner.discover_csv_files()
    combiner.load_datasets()

    combined_df = combiner.create_combined_dataset(percentages)
    combiner.save_combined_dataset(combined_df, output_name)

    return combined_df

def interactive_setup():
    """Interactive setup for users who want guided configuration"""
    print("🔧 Interactive Dataset Combiner Setup")
    print("="*50)

    # Get folder path
    folder_path = input("Enter Google Drive folder path: ").strip()

    # Initialize and discover files
    combiner = DatasetCombiner(folder_path)
    combiner.mount_drive()
    files = combiner.discover_csv_files()
    combiner.load_datasets()

    # Get percentages for each file
    percentages = {}
    print(f"\nEnter percentage for each file (must sum to 100):")

    for file in files:
        while True:
            try:
                perc = float(input(f"  {file}: "))
                percentages[file] = perc
                break
            except ValueError:
                print("    Please enter a valid number")

    # Validate total
    total = sum(percentages.values())
    if abs(total - 100) > 0.01:
        print(f"Warning: Total is {total}%, not 100%")

    # Get output filename
    output_name = input("Output filename (default: combined_dataset_2.csv): ").strip()
    if not output_name:
        output_name = "combined_dataset_14_July.csv"

    # Create and save
    combined_df = combiner.create_combined_dataset(percentages)
    combiner.save_combined_dataset(combined_df, output_name)

    return combined_df

# Run the main function
if __name__ == "__main__":
    main()

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully!
Discovering CSV files in: /content/drive/MyDrive/FYP/
Found 24 CSV files:
  1. web-ids23_dos_http.csv
  2. web-ids23_bruteforce_https.csv
  3. web-ids23_bruteforce_http.csv
  4. web-ids23_benign.csv
  5. anomaly_detection_dataset.csv
  6. web-ids23_dos_https.csv
  7. web-ids23_ftp_version.csv
  8. web-ids23_ftp_login.csv
  9. web-ids23_hostsweep_sn.csv
  10. web-ids23_revshell_http.csv
  11. web-ids23_revshell_https.csv
  12. web-ids23_smtp_enum.csv
  13. web-ids23_smtp_version.csv
  14. web-ids23_sql_injection_http.csv
  15. web-ids23_sql_injection_https.csv
  16. web-ids23_ssh_login.csv
  17. web-ids23_ssh_login_successful.csv
  18. web-ids23_ssrf_http.csv
  19. web-ids23_ssrf_https.csv
  20. web-ids23_xss_http.csv
  21. web-ids23_hostsweep_Pn.csv
  22. web-ids23_xss_https.csv
  23. combined_securi